# 🗣️ TTS 챕터 종합 정리 노트 (3주차 + 5-S/6-S 연결)

> **생성형 AI 기반 음성 에이전트 개발 과정 · TTS 파트 리뷰**
> `TTS/` 폴더의 실습 노트북 14개를 하나로 통합한 **복습·재사용용 노트**입니다.

| 항목 | 내용 |
|---|---|
| 대상 노트북 | `TTS/` 14개 (3-1 개요 ~ 6-S 스트리밍 TTS) |
| 노트의 목적 | ① **선행 지식** ② **함수/클래스** 정의·주석 ③ **실험 진행 방법** ④ **효율적 설계 아키텍처** |
| 실행 환경 | Google Colab(T4) 실습 기준 / **macOS(Apple Silicon) 실행 가이드 포함** |
| 과정 공통 표준 | TTS 계약(`TTS_CONTRACT_KEYS`) · 표준 발화 `tts_001~005` · TTS 예산 **0.5s** · **3축 판정**(한국어·지연·라이선스) |

> ⚙️ **실행 안내** — 이 노트의 코드 셀은 **GPU·모델·네트워크 없이 실행되는** 정규화·계약·시뮬레이션·어댑터만 모았습니다.
> 모델 합성이 필요한 함수는 시그니처+설명으로 요약했습니다 (2장에서 ✅/📄로 구분). 위에서 아래로 실행하세요.


## 📑 목차
| 장 | 내용 |
|---|---|
| **0** | 노트북 로드맵 (14개 중요도·의존 관계·macOS 지원·'카드' 구조 해설) |
| **1** | 실험에 필요한 선행 지식 (11개 주제) |
| **2** | 함수/클래스 정의 및 주석 (실행 코드 ✅ + 요약 📄) |
| **3** | 실험 진행 방법 (노트북별 가이드 + macOS 실습법) |
| **4** | 효율적 설계를 위한 아키텍처 |


# 0. 노트북 로드맵 🗺️

## 0-1. 14개 노트북 한눈에

| 노트북 | 주제 | 중요도 | macOS 실행 | 코멘트 |
|---|---|---|---|---|
| **3-1** | TTS 시스템 개요 (3단 구조·정규화·G2P·보코더·계약) | ★★★ 코어 | ✅ (gTTS/Mock) | 모든 개념의 출발점 |
| **3-2** | 경량 한국어 TTS (Supertonic 3, ONNX CPU) | ★★★ 코어 | ✅ | 계약 방어·p95·TTFB |
| **3-3** | 고품질 다국어 TTS (XTTS v2) | ★★★ 코어 | ⚠️ GPU+라이선스 | 라이선스 게이트·다국어 |
| 5-S | 응답 시나리오 설계 (대화층) | ★★ 파이프라인 | ✅ (MockLLM) | 5주차 LLM 통합 서막 |
| 6-S | 스트리밍 TTS 적용 (의사-스트리밍·프리버퍼) | ★★ 파이프라인 | ✅ 시뮬레이션 | 6주차 실시간 서막 |
| 3-F | FishAudio S1-mini (LLM 기반·이중 게이트) | ★★ 방법론 | ✖ 비상업 | **왕복 CER 탄생** |
| 3-L | MeloTTS ("CPU 실시간" 주장 검증) | ★★ 방법론 | ✅ | VITS 계열·핀 우회 |
| 3-K | Kokoro-82M (9개 언어 전수) | ★★ 방법론 | ✅ | 테이블 판정·G2P 3원 |
| 3-M | MOSS-TTS v1.5 (3축 종합 판정) | ★ 방법론 | ✖ GPU | 3주차 카드 최종장 |
| 3-Q | Qwen3-TTS (CustomVoice) | ★ 방법론 | ✖ GPU | **device-side assert 사건** |
| 3-S | Meta MMS-TTS (1,100개 언어) | ★ 방법론 | ✅ (CPU) | 저장소=지원·**입력 계약** |
| 3-G | GPT-SoVITS (저장소형·커널 교체) | ★ 방법론 | ✖ | sr는 체크포인트의 속성 |
| 3-D | F5-TTS 한국어 구조대 (체크포인트 부검) | ★ 방법론 | ✖ GPU | 문서 0바이트의 교훈 |
| 3-ST | StyleTTS 2 (Kokoro의 조상) | ★ 방법론 | ✖ | **부품 계층** 미지원 |

> **'카드' 노트북의 정체**: 3-D~3-ST(9개)는 모두 **동일한 표준 장비**를 엔진마다 반복합니다 —
> 설치(설계된 재시작) → 계약 → 합성 → 역전사 CER → 변환 → 지연 → 라이선스 판정 → "오늘의 세 문장".
> 이 노트에서는 **방법론은 1번**(1장), **엔진 비교는 1개의 통합 카드 표**(2.8)로 압축했습니다.

## 0-2. 학습 흐름 (의존 관계)

```
3-1(구조·정규화·G2P·보코더·TTS 계약) ──→ 3-2(경량·p95·TTFB) ──→ 3-3(고품질·라이선스)
   │
   ├── 표준 장비: 역전사 CER(3-F) · 계약(3-1) · 3축 판정(한국어/지연/라이선스)
   │       └── 엔진 카드: 3-Q 3-F 3-K 3-L 3-M 3-S 3-G 3-D 3-ST
   │
   └── 5-S(대화층: 응답 계약·분류기·가드레일) ──→ 6-S(스트리밍 TTS: 프리버퍼·오버랩)
```

## 0-3. 과정 공통 '표준' 4종 (모든 실험의 무대)

1. **TTS 계약 `TTS_CONTRACT_KEYS`** — `{audio, sr, duration_ms, latency_ms, engine}`. `sr != 16000` → 즉시 예외.
   "좋은 모델도 우리 계약을 모른다" — 14개 세션 내내 이 방어가 반복됩니다.
2. **표준 발화 `tts_001~005`** — 콜센터 응대 문장(금액·전화번호·개수 포함). ASR의 `utt_001~005`와 대칭.
3. **가혹조건·지연 예산** — 왕복 1.5s 중 **TTS 몫 0.5s**(작업 가정, 5주차 재협상). 판정은 평균이 아니라 **p95**.
4. **3축 판정** — 한국어(합성→역전사 CER) · 지연(실측) · 라이선스(문서). **축마다 판정 도구가 다르다**.


## 📖 0-A. 용어 사전 & 배경 지식 — 이 노트를 처음 읽는 사람을 위한 지도

> **이 노트를 처음 공부하는 방법**: ① 0-A 용어사전 훑기 → ② 1장 선행 지식(개념의 이유) →
> ③ 2장 함수 실행하며 "검증 통과 ✅" 눈으로 확인. 모르는 단어는 여기로 돌아오세요.

### A. TTS 기본 구조 — "텍스트가 목소리가 되기까지"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| 텍스트 정규화 (NSW) | "3,000원"→"삼천 원", "010-1234"→"공일공" 변환 | 읽기 전에 숫자·기호를 '말로 읽을 말'로 |
| G2P | 글자(Grapheme) → 발음(Phoneme) 변환 | 한국어는 표기와 발음이 다름(연음·된소리) |
| 연음 (liaison) | '요금은' → '요그믄' — 종성이 다음 초성으로 이동 | 한국어 TTS 품질의 핵심 규칙 |
| 멜 스펙트로그램 | 주파수 축을 사람 귀에 맞춘 '중간 언어' | ASR/VC와 공유하는 TTS 내부 표현 |
| 보코더 | 멜 그림 → 실제 파형 | "그림을 소리로 되살리는" 마지막 단계 |
| 음소 (phoneme) | 소리의 최소 단위 (ㄱ,ㅏ,ㅁ…) | 모델이 배우는 '발음 알파벳' |

### B. TTS 품질·지연 지표
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| p95 / TTFB | 95번째 백분위 지연 / 첫 오디오 도달 시간 | 사용자 체감 지연의 진짜 값 |
| RTF | 처리시간 / 오디오 길이. 1 미만 = 실시간 | 실시간 대화 가능 여부 |
| CER (왕복) | 합성음 → ASR → 다시 텍스트 → 오류율 | "합성음이 잘 들리는가"의 객관적 심사 |
| 프리버퍼 | 재생 시작 전 미리 쌓아두는 오디오 버퍼 | 언더런(끊김) 방지 |
| 언더런 (underrun) | 버퍼가 비어 재생이 끊기는 현상 | 실시간 음성의 적절한 무게감 |

### C. 아키텍처 세대 — "지연의 물리학"
| 용어 | 한 줄 뜻 | 왜 중요한가 |
|---|---|---|
| AR (자기회귀) | 토큰을 한 개씩 순서대로 생성 | 품질 좋지만 느림 |
| NAR (비자기회귀) | 여러 토큰을 한 번에 생성 | 빠르지만 품질 트레이드오프 |
| 대화층 | TTS 위에 얹는 응답 계약·가드레일 | "무엇을 말할지"를 정제하는 층 |
| 가드레일 | 위험·금지 표현을 막는 규칙 | 안전한 음성 에이전트의 필수 장치 |

### D. 배경 지식 — 이 챕터가 왜 존재하는가
이 과정(음성 에이전트)에서 **TTS는 "말하는 입"**입니다. ASR(귀)이 들은 것을 LLM(뇌)이 생각해서
TTS가 소리로 내보냅니다. 이 노트가 다루는 핵심:
1. **"좋은 모델도 우리 계약을 모른다"** — 모델은 24kHz·스테레오로 뱉을 수 있지만, 우리 시스템은
   **16kHz·모노·float32**를 요구한다. 어댑터가 계약으로 접어야 한다 (3-1 계약 위반 쌍방향).
2. **한국어 특수성** — 텍스트 정규화·G2P·연음이 영어와 완전히 다르다. "3,000원"을 못 읽으면
   품질은 모델이 아니라 정규화기가 결정한다.
3. **지연이 곧 체감 품질** — 대화에서 200ms는 참을 수 있지만 1초는 참지 못한다. p95·프리버퍼·언더런을
   계측하고, AR→NAR 세대 변화와 스트리밍 합성이 그 해법이다.


### 0-B. 2장 함수 지도 — 어떤 셀이 무슨 역할인지 미리 보기
| 셀 | 함수/클래스 | 역할 한 줄 | 핵심 개념 |
|---|---|---|---|
| 2.0 | `num_to_sino`·`normalize_money`·`normalize_phone`·`normalize_count` | 한국어 텍스트 정규화 | 숫자·전화번호·수량 → 말로 읽을 말 |
| 2.1 | `decompose`·`compose`·`apply_liaison` | 한글 G2P (연음) | 유니코드 산술 + 연음 법칙 |
| 2.2 | `TTS_CONTRACT_KEYS`·`MockTTS`·`validate_tts_contract` | TTS 계약 + Mock 엔진 | 16kHz 모노 float32 강제 |
| 2.3 | `pctl`·`split_sentences`·`ttfb_stats` | 지연 통계 + 문장 분할 | p95·TTFB |
| 2.4 | 응답 계약·분류기·LLM 인터페이스·가드레일 | 대화층 | 무엇을 말할지 정제 |
| 2.5 | 스트리밍 TTS 코어 | 프리버퍼·언더런·오버랩 | 실시간 합성 |
| 2.6 | macOS 어댑터 | Apple Silicon 실행 | 로컬 실습 |


# 1. 실험에 필요한 선행 지식 🧠

## 1-1. TTS 3단 구조 (3-1) — 텍스트가 목소리가 되기까지

```
텍스트 ──→ ① 텍스트 분석 ──→ ② 음향 모델 ──→ ③ 보코더 ──→ 파형
          정규화·G2P·운율      문자 → 멜-스펙트로그램   멜 → 오디오 샘플
```

| 단 | 역할 | AICC 관점 |
|---|---|---|
| ① 텍스트 분석 | NSW 정규화(금액·전화·개수) + G2P(표기→발음) | **품질의 절반은 여기서 결정** — "요금은 35,000원입니다"를 정규화 없이 넣으면 잘못 읽음 |
| ② 음향 모델 | 문자 → 멜-스펙트로그램 (TTS의 **중간 언어**) | 2-F에서 ASR 입력이었던 바로 그 표현 — 방향만 반대 |
| ③ 보코더 | 멜 → 파형 (Griffin-Lim ~ 신경 보코더) | Griffin-Lim은 위상 손실로 금속성 울림 → 신경 보코더가 품질·RTF 동시 개선 |

> 4주차(보이스 클로닝)의 출발점: **② 음향 모델이 '누구의 목소리인가'를 결정**하는 단.

## 1-2. 한국어 텍스트 정규화 — NSW 3종 (3-1)

| NSW | 원문 | 낭독 | 수사 체계 |
|---|---|---|---|
| 금액 | 35,000원 | 삼만 오천 원 | **한자어** (만 단위 묶음 — 서양 3자리와 어긋남) |
| 전화번호 | 010-1234-5678 | 공일공, 일이삼사, 오육칠팔 | 한 자리씩, 0은 '공' |
| 개수 | 상담원 3명 | 상담원 세 명 | **고유어** (한/두/세) |

- **규칙 순서가 곧 버그다**: 전화번호 → 금액 → 개수 순 (금액이 먼저면 전화번호 뒷자리를 먹는다).
- `normalize_phone`의 함정: 한글도 `\w`라서 `\b`가 조사('5678**로**')에서 안 생긴다 → 숫자 lookaround `(?<!\d)…(?!\d)`로 해결.
- **규칙 기반의 본질적 한계**: `1544-0000`(대표번호, 0으로 시작 안 함)은 패턴 밖 → 규칙은 예상한 것만 잡는다.

## 1-3. 한글 G2P 맛보기 — 유니코드 산술 + 연음 (3-1)

- 음절 = `0xAC00 + 초성×588 + 중성×28 + 종성` — 11,172자가 한 줄 산술로 분해/조합.
- **연음 법칙**: 종성 + 초성ㅇ → 종성이 다음 음절 초성으로 이동. "요금은" → [요그믄].
- 종성 ㅇ은 [ŋ]이라 이동하지 않는다("강아지" 유지) — g2pk 라이브러리의 영역은 실전에서 사용.
- 왜 배우나: 4주차 클로닝 TTS들의 G2P 품질이 제각각이라, 잘못 읽는 단어를 발음 표기로 '치료'하는 일이 실무에서 빈번.

## 1-4. 멜 = 중간 언어, 보코더의 위상 문제 (3-1 · 2-F)

- **ASR**: 파형 → 멜 → 텍스트 / **TTS**: 텍스트 → 멜 → 파형 — 같은 다리를 반대 방향으로.
- 멜은 **크기(magnitude)만** 담고 **위상(phase)을 버린다** → Griffin-Lim(1984)은 "그럴듯한 위상 반복 추정".
- Griffin-Lim 32회 반복은 CPU에서 RTF ≥ 1이 나오기 쉬움 → **품질 이전에 실시간 파이프라인 입장이 불가** → 신경 보코더(HiFi-GAN 등)가 표준.


## 1-5. TTS 계약·어댑터 — 계약 위반은 쌍방향이다 (3-1~3-3, 카드 전부)

**엔진마다 출력 규격이 제각각** — 어댑터가 혼돈을 계약 뒤로 격리합니다. **변환 책임은 파이프라인 쪽**.

| 엔진 | 원출력 | 반환 타입 | sr 위반 여부 |
|---|---|---|---|
| gTTS (3-1) | 24kHz | mp3 파일 | ❌ |
| Supertonic (3-2) | 44.1kHz | ndarray | ❌ |
| XTTS (3-3) | 24kHz | 리스트 | ❌ |
| Qwen3-TTS (3-Q) | 24kHz | ndarray | ❌ |
| MeloTTS (3-L) | 44.1kHz | ndarray | ❌ |
| MOSS-TTS (3-M) | 24kHz | ndarray | ❌ |
| S1-mini (3-F) | 44.1kHz | ndarray | ❌ |
| Kokoro (3-K) | 24kHz | float32 | ❌ |
| **MMS-TTS (3-S)** | **16kHz** | ndarray | ✅ **처음으로 일치** (1/7) |

> **통계 교훈 (3-S)**: "원시 출력 sr 일치 0/6"은 법칙이 아니라 표본이었다 — 1/7이 된 날, 리샘플 없는 변환 셀(캐스팅만)이 첫 등장. **단계가 줄어도 피크 가드는 남는다.**
> **입력측 계약 (3-S)**: MMS는 한국어에 `is_uroman=True` — 입력이 로마자여야 함. 위반 시 **경고 한 줄과 조용한 문자 소실** → 로그를 읽어라.

## 1-6. 품질·지연 지표 (3-1 ~ 3-3, 5-S, 6-S)

| 지표 | 정의 | 판정 기준 |
|---|---|---|
| **역전사 CER** | 합성음 → faster-whisper → 원문과 CER | TTS↔ASR **왕복** 표준 장비 (3-F) — 귀 대신 도구로 |
| **MOS** | 1~5 주관 청취 | 산업 표준이지만 사람 필요 |
| **TTFB** | 첫 오디오까지 시간 = 비스트리밍의 전체 지연 | 문장 분할로 공략 (3-2) |
| **p95 지연** | 꼬리 지연 — 20명 중 1명의 최악 대기 | 예산 판정은 **평균이 아니라 p95** (3-2) |
| **RTF** | 처리시간 ÷ 오디오길이 | <1.0이어야 실시간 후보 |
| **예산 상속** | 1.5s = ASR 0.5 + TTS 0.5 + **대화층 0.5** | 5-S: 층마다 예산이 상속되고 다시 분할 |

## 1-7. 라이선스 게이트 — "동의는 코드 한 줄, 위반은 계약 문제" (3-3 ~ 3-ST)

| 라이선스 | 모델 | 상업 배포 | 비고 |
|---|---|---|---|
| **CPML** | XTTS v2 | ❌ (모델·출력물 모두) | 3-3의 게이트 |
| **CC-BY-NC-SA-4.0** | S1-mini | ❌ | HF gated repo + 비상업 = **이중 게이트** (3-F) |
| **CC-BY-NC-4.0** | MMS-TTS | ❌ | 무게이트 (3-S) |
| Apache-2.0 | MOSS v1.5, F5-TTS-ko, Kokoro | ✅ (조건부) | F5-ko는 **데이터 계보 미공개** 회색지대 (3-D) |
| MIT | Supertonic, MeloTTS, GPT-SoVITS, StyleTTS2 | ✅ | StyleTTS2는 MIT 아래 **고지 의무 + GPL 의존** (3-ST) |

> **라이선스 축은 문서 축이다** (3-D): 문서가 0바이트인 모델은 라이선스 라벨도 0바이트만큼만 믿을 수 있다. AICC 납품 시 라이선스 실사가 첫 질문.

## 1-8. 지원 판정 4종 + 미지원 유형학 (3-K ~ 3-ST)

**"한국어 지원?"을 판정하는 방법 4종** — 어느 것이든 '기계가 확인 가능한 사실' 위에 서야 함:
① **테이블** (3-K Kokoro `LANG_CODES`, 3-L MeloTTS 'KR') ② **실험·왕복 CER** (3-F S1-mini — 코드가 침묵하면 실험이 판정) ③ **저장소의 존재** (3-S MMS `facebook/mms-tts-kor`, ISO 639-3 함정: `ko`가 아니라 `kor`) ④ **부품 조합** (3-ST StyleTTS2)

**미지원에도 지층이 있다** (3-ST) — 깊을수록 문이 비싸진다:

| 세션 | 미지원 메커니즘 | 고치는 문 |
|---|---|---|
| 3-D F5 | **어휘 밖** — 한글이 인덱스 0으로 뭉개지는 침묵 실패 | 어휘 수정 |
| 3-G GPT-SoVITS | **버전의 함수** — v2 테이블부터 'ko' | 버전 선택 |
| 3-ST StyleTTS 2 | **부품 계층** — espeak는 아는데 PL-BERT가 모른다 | PL-BERT-ko 사전학습부터 |

## 1-9. 아키텍처 세대 — AR vs NAR, 지연의 물리학 (3-2 ~ 3-M)

| 세대 | 구조 | 지연 성질 | 예 |
|---|---|---|---|
| **VITS 계열 (NAR)** | 비자기회귀, 한 번에 합성 | 짧고 **거의 평평** | Supertonic(3-2), MeloTTS(3-L), MMS(3-S), StyleTTS2(3-ST) |
| **LLM-TTS (AR)** | 자기회귀, 토큰 생성 | 길수록 선형 증가 | Qwen3-TTS(3-Q), MOSS(3-M), S1-mini(3-F) |
| **혼혈** | GPT(AR) + SoVITS(VITS) | AR 성질이 지배 | GPT-SoVITS(3-G) |

> Week 5 재협상의 실질은 **두 지연 곡선 사이의 선택**. 3주차 결론: *한국어·지연·라이선스 세 축을 동시에 만족하는 카드가 없었다* (3-M 최종표).


## 1-10. macOS(Apple Silicon) 실행 가이드 🍎 (추가 조사)

> TTS는 **CPU 실시간**을 내세우는 경량 엔진이 많아 macOS 친화적입니다. ONNX Runtime 기반이면 거의 그대로 동작합니다.

| 런타임 | 설치 | macOS 실행 | 한국어 | 라이선스 | 비고 |
|---|---|---|---|---|---|
| **Supertonic 3** | `pip install supertonic` | ONNX CPU 실시간 | ✅ 31개 언어 | MIT | 3-2의 엔진 그대로. 44.1kHz 출력 → 계약 어댑터 필수 |
| **MeloTTS** | `pip install melotts` | `TTS(language='KR', device='cpu')` — CLI는 `--device mps`도 지원 | ✅ KR | MIT | 3-L의 "CPU 실시간" 주장 — MPS로 더 빠름 |
| **Kokoro-82M** | `pip install kokoro` | CPU 완주 가능 | ❌ v1.0 9개 언어 | Apache-2.0 | 3-K — G2P는 misaki/espeak-ng |
| **MMS-TTS** | `pip install transformers` | CPU (VITS 세대) | ✅ kor | CC-BY-NC | 3-S — 한국어는 `is_uroman=True` (로마자 입력) |
| **gTTS / Mock** | `pip install gTTS` | 클라우드(네트워크) | ✅ | — | 3-1 기준선 |
| **sherpa-onnx** (추천 신규) | `pip install sherpa-onnx` | ONNX 통합 런타임 — macOS·엣지 | ✅ Supertonic/Kokoro 등 | 모델별 | 강의 외 도구 — **Supertonic·Kokoro·Piper·MeloTTS를 한 API로** + 스트리밍 지원 |
| XTTS v2 / Qwen3 / MOSS / S1 / F5 / StyleTTS2 | — | ⚠️ GPU 전용(비공식 MPS) | — | CPML/NC | 3주차 카드 — macOS보다 Colab T4 |

**macOS 추천 조합**
1. **기본**: Supertonic 3 (CPU, 한국어, MIT) — 3-2 실습 그대로.
2. **고속/다용도**: sherpa-onnx — Supertonic·Kokoro 등을 한 API로, 실시간·스트리밍 모두.
3. **품질 상한선**: gTTS(클라우드) vs MeloTTS(로컬, MPS) 비교 — p95 예산으로 판정.
4. **5-S/6-S**: 전부 시뮬레이션이라 코드 수정 없음.

## 1-11. 설치·재시작 원칙 — '설계된 재시작' 3형 (2-M ~ 3-G)

| 형태 | 세션 | 본질 |
|---|---|---|
| **사고** | 2-M | fairseq2 + 이미 로드된 torch → 재시작 루프 |
| **절차** | 3-F | numpy≤1.26.4 핀이 런타임 강등 요구 → **설치 전에 METADATA를 읽고** 재시작 지점을 노트북에 미리 박음 |
| **커널 교체** | 3-G | condacolab으로 py3.12 → py3.10 — "세션 다운 배너"가 **정상 동작** |

> 같은 현상, 다른 경험. 차이는 단 하나 — **메타데이터를 먼저 읽었는가.** 설치 셀은 모든 import보다 먼저, 단 한 번만.


# 2. 실험에 필요한 함수/클래스 정의 및 주석 🔧

> **✅ = 실행 가능한 코드 셀** (GPU·모델·네트워크 불필요) · **📄 = 시그니처+설명 요약** (모델/API 필요)
> 위에서 아래로 실행하세요. 각 셀은 자가 점검(assert)을 포함합니다.

| 번호 | 항목 | 실행 |
|---|---|---|
| 2.0 | 한국어 정규화기: `num_to_sino`·`normalize_money/phone/count`·`normalize_aicc` | ✅ |
| 2.1 | 한글 G2P: `decompose`·`compose`·`apply_liaison` | ✅ |
| 2.2 | TTS 계약: `TTS_CONTRACT_KEYS`·`validate_tts_contract`·`mock_synthesize` | ✅ |
| 2.3 | 지연 통계·문장 분할: `latency_stats`·`split_sentences` | ✅ |
| 2.4 | 대화층 (5-S): 응답 계약·`classify_rule`·`MockLLM`·`generate_guarded` | ✅ |
| 2.5 | 스트리밍 코어 (6-S): `simulate_playback`·`min_prebuffer`·`overlap_pipeline` | ✅ |
| 2.6 | macOS 어댑터: `mac_melo_synthesize`·`mac_supertonic_synthesize`·`mac_sherpa_onnx` | ✅ |
| 2.7 | 왕복 CER 표준 장비: `make_tts_record`·`roundtrip_cer` | 📄 |
| 2.8 | 엔진 9종 통합 카드 표 (계약 위반·라이선스·CER·지연) | 📄 |
| 2.9 | 엔진별 어댑터 요약 (sr/dtype 위반 유형) | 📄 |


In [ ]:
# ═══ 2.0 한국어 텍스트 정규화기 (3-1) — AICC 품질의 절반 ═══
# ▶ 정규화 3종은 '숫자·기호를 발음으로' 바꾼다: 화폐(원)·전화번호(공일공)·수량(세 개).
#   한국어는 4자리(만 단위) 묶음 — 서양 3자리 콤마와 어긋나는 것이 함정이다.
#   ⚠️ \b(단어 경계)는 한글 조사에서 안 생긴다 → 숫자 lookaround로 회피 (주석의 교훈).
import re

DIGITS = "영일이삼사오육칠팔구"
SMALL_UNITS = ["", "십", "백", "천"]
BIG_UNITS = ["", "만", "억", "조"]

def num_to_sino(n):
    """정수 → 한자어 수사. 관행: 맨 앞 '일만'은 '만'으로 (일만 오천 → 만 오천).
    한국어는 4자리(만 단위) 묶음 — 서양의 3자리 콤마와 어긋난다."""
    if n == 0:
        return "영"
    groups = []
    while n > 0:
        groups.append(n % 10000)
        n //= 10000
    parts = []
    for gi in range(len(groups) - 1, -1, -1):
        g = groups[gi]
        if g == 0:
            continue
        if gi == 1 and g == 1 and len(groups) == 2:
            parts.append("만")          # '일만 원'보다 '만 원'이 낭독 관행
            continue
        s = ""
        for pos in range(3, -1, -1):
            d = (g // 10 ** pos) % 10
            if d == 0:
                continue
            s += ("" if (d == 1 and pos > 0) else DIGITS[d]) + SMALL_UNITS[pos]
        parts.append(s + BIG_UNITS[gi])
    return " ".join(parts)

def normalize_money(text):
    def repl(m):
        return num_to_sino(int(m.group(1).replace(",", ""))) + " 원"
    return re.sub(r"(\d[\d,]*)\s*원", repl, text)

def normalize_phone(text):
    """쉼표는 TTS 운율(pause) 힌트 — 붙여 읽으면 자리수를 놓친다.
    ⚠️ 한글도 \\w라서 \\b가 조사('5678로')에서 안 생긴다 → 숫자 lookaround로 해결."""
    def repl(m):
        read = lambda b: "".join("공" if c == "0" else DIGITS[int(c)] for c in b)
        return ", ".join(read(b) for b in m.group(0).split("-"))
    return re.sub(r"(?<!\d)0\d{1,2}-\d{3,4}-\d{4}(?!\d)", repl, text)

NATIVE = {1: "한", 2: "두", 3: "세", 4: "네", 5: "다섯",
          6: "여섯", 7: "일곱", 8: "여덟", 9: "아홉", 10: "열"}

def normalize_count(text):
    def repl(m):
        n, counter = int(m.group(1)), m.group(2)
        return f"{NATIVE[n]} {counter}" if n in NATIVE else m.group(0)
    return re.sub(r"(\d+)\s*(개|명|분|건|가지)", repl, text)

def normalize_aicc(text):
    """⚠️ 순서 중요: 전화번호 → 금액 → 개수 (금액이 먼저면 전화번호 뒷자리를 먹는다)."""
    return normalize_count(normalize_money(normalize_phone(text)))

# ── 자가 점검 ──
assert num_to_sino(35000) == "삼만 오천"
assert num_to_sino(10000) == "만"          # 관행 규칙
assert num_to_sino(100000000) == "일억"     # 억부터는 '일' 유지
assert num_to_sino(2026) == "이천이십육"
assert normalize_phone("010-1234-5678로 연락드리겠습니다") \
       == "공일공, 일이삼사, 오육칠팔로 연락드리겠습니다"   # 조사 '로'에도 매칭
assert normalize_aicc("010-1234-5678 고객님, 미납액 15,000원과 문의 2건이 있습니다") \
       == "공일공, 일이삼사, 오육칠팔 고객님, 미납액 만 오천 원과 문의 두 건이 있습니다"
print("한국어 정규화기 검증 통과 ✅")
print("예: '요금은 35,000원, 상담원 3명' →", normalize_aicc("요금은 35,000원, 상담원 3명"))
print("⚠️ tts_005의 '1544-0000'(대표번호)은 패턴 밖 — 규칙은 예상한 것만 잡는다.")


In [ ]:
# ═══ 2.1 한글 G2P 맛보기 — 유니코드 산술 + 연음 법칙 (3-1) ═══
# ▶ decompose/compose = 한글 음절 ↔ (초성,중성,종성) 왕복 변환. 유니코드 0xAC00 기준.
#   apply_liaison: 종성 + 초성ㅇ → 종성이 다음 초성으로 이동. '요금은'→'요그믄'.
#   ⚠️ 종성 ㅇ은 [ŋ]이라 이동하지 않는다 ('강아지'≠'가ㅇ아지').
CHO = list("ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ")            # 19
JUNG = list("ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ")       # 21
JONG = [""] + list("ㄱㄲ") + ["ㄳ", "ㄴ", "ㄵ", "ㄶ", "ㄷ", "ㄹ",
        "ㄺ", "ㄻ", "ㄼ", "ㄽ", "ㄾ", "ㄿ", "ㅀ", "ㅁ", "ㅂ",
        "ㅄ"] + list("ㅅㅆㅇㅈㅊㅋㅌㅍㅎ")                        # 28 (없음 포함)

def decompose(ch):
    """한글 음절 → (초성, 중성, 종성) 인덱스. 음절 밖이면 None."""
    code = ord(ch) - 0xAC00
    if not (0 <= code < 11172):
        return None
    return code // 588, (code // 28) % 21, code % 28

def compose(cho, jung, jong):
    """(초성, 중성, 종성) 인덱스 → 한글 음절. 왕복 무손실."""
    return chr(0xAC00 + cho * 588 + jung * 28 + jong)

# 종성 ㅇ은 [ŋ]이라 이동하지 않는다 (강아지 ≠ 가ㅇ아지)
LIAISON = {ji: CHO.index(jc) for ji, jc in enumerate(JONG)
           if jc in CHO and jc != "ㅇ"}

def apply_liaison(text):
    """연음 법칙: 종성 + 초성ㅇ → 종성이 다음 음절 초성으로 이동.
    '요금은' → '요그믄'. 겹받침 등 복잡 규칙은 g2pk 라이브러리의 영역."""
    chars = list(text)
    for i in range(len(chars) - 1):
        a, b = decompose(chars[i]), decompose(chars[i + 1])
        if a is None or b is None:
            continue
        (cho_a, jung_a, jong_a), (cho_b, jung_b, jong_b) = a, b
        if jong_a in LIAISON and cho_b == CHO.index("ㅇ"):
            chars[i] = compose(cho_a, jung_a, 0)                       # 종성 떼고
            chars[i + 1] = compose(LIAISON[jong_a], jung_b, jong_b)    # 다음 초성에
    return "".join(chars)

# ── 자가 점검 ──
c, j, g = decompose("금")
assert compose(c, j, g) == "금"
for src, expect in [("요금은", "요그믄"), ("옷을", "오슬"),
                    ("음악", "으막"), ("강아지", "강아지"), ("안녕하세요", "안녕하세요")]:
    got = apply_liaison(src)
    assert got == expect, f"{src}: {got} != {expect}"
    print(f"  {src} → {got if got != src else '= (무변화)'}")
print("한글 G2P 검증 통과 ✅")


In [ ]:
# ▶ 데모 — 텍스트 정규화 & 연음, 눈으로 확인하기 (초보자용)
# '모델이 못 읽는 글'을 정규화기가 '말로 읽을 글'로 바꾸는 과정을 한 줄씩 본다.

samples = ["총 30,000 원 결제되었습니다", "상담사 010-1234-5678 연락드릴게요",
           "배송은 3 일 소요됩니다", "요금은 확인 후 알려드리겠습니다"]

for s in samples:
    step1 = normalize_money(normalize_phone(normalize_count(s)))
    print(f"입력 : {s}")
    print(f"정규화 후: {step1}")
    print(f"연음  후: {apply_liaison(step1)}")
    print()

# 스스로 점검: 정규화 → 연음이 항상 안전하게 끝나는가
assert "삼만 원" in normalize_money("30,000 원")
assert "공일공" in normalize_phone("010-1234-5678")
assert apply_liaison("요금은") == "요그믄"
print("데모 통과 ✅ — TTS는 '무엇을 말할지'가 정해져야 소리를 낸다")


In [ ]:
# ═══ 2.2 TTS 계약 + Mock 엔진 (3-1) — 좋은 모델도 우리 계약을 모른다 ═══
# ▶ 핵심: '좋은 모델도 우리 계약을 모른다' — MockTTS는 16kHz 모노 float32를 강제로 지킨다.
#   validate_tts_contract가 위반하면 즉시 실패(시끄러운 실패) — 조용히 넘어가면 나중에 터진다.
import os, time
import numpy as np

SR = 16000                                  # 16kHz mono PCM_16 — 과정 공통 규격
TTS_CONTRACT_KEYS = {"audio", "sr", "duration_ms", "latency_ms", "engine"}

def validate_tts_contract(r):
    """계약 검증 — 위반 시 AssertionError (고치지 않는다, 시끄럽게 막는다).
    3-2에서 Supertonic의 44.1kHz를 경계에서 차단한 바로 그 방어."""
    missing = TTS_CONTRACT_KEYS - set(r.keys())
    if missing:
        raise AssertionError(f"TTS 계약 위반 — 누락 키: {sorted(missing)}")
    if r["sr"] != SR:
        raise AssertionError(f"샘플레이트 규격 위반: {r['sr']} != {SR}")
    if getattr(r["audio"], "ndim", 1) != 1:
        raise AssertionError("오디오는 mono(1차원)여야 함")
    if r["duration_ms"] <= 0:
        raise AssertionError("duration_ms는 양수여야 함")
    return True

def mock_synthesize(text, uid):
    """리허설/네트워크 차단용 대역: 음절 리듬을 흉내 낸 합성음 (seed 결정적).
    지연을 길이에 비례시켜 실제 비스트리밍 TTS의 길이-지연 관계를 보존한다."""
    rng = np.random.default_rng(42 + len(text))
    dur = 0.16 * len(text.replace(" ", "")) + 0.5        # 한국어 ≈ 6음절/초
    t = np.arange(int(SR * dur)) / SR
    env = 0.5 * (1 + np.sin(2 * np.pi * 5.5 * t))        # 음절 단위 진폭 변조
    y = (0.15 * env * np.sin(2 * np.pi * 180 * t)
         + 0.03 * env * np.sin(2 * np.pi * 360 * t)
         + 0.005 * rng.normal(size=len(t))).astype(np.float32)
    n_syl = len(text.replace(" ", ""))
    latency = 120.0 + 9.0 * n_syl + float(rng.uniform(-20, 20))
    return {"audio": y, "sr": SR, "duration_ms": len(y) / SR * 1000.0,
            "latency_ms": latency, "engine": "mock"}

# ── 자가 점검 ──
ok = mock_synthesize("안녕하세요 고객님", "t1")
assert validate_tts_contract(ok)
bad = dict(ok); bad["sr"] = 44100          # 3-2의 계약 방어 재현
try:
    validate_tts_contract(bad); raise SystemExit("방어 실패!")
except AssertionError as e:
    print("계약 방어 확인 ✅ →", e)
print("TTS 계약·Mock 엔진 검증 통과 ✅ (44.1kHz를 경계에서 차단)")


In [ ]:
# ═══ 2.3 지연 통계 + 문장 분할 (3-2 · 6-S) — p95와 TTFB ═══
# ▶ p95: '대부분의 호출은 얼마나 걸리는가' — 평균은 꼬리 지연을 가린다.
#   split_sentences: 긴 문장을 자르면 첫 오디오(TTFB)가 빨라진다 — 지연 최적화의 기본.
import re
import numpy as np

TTS_BUDGET_MS = 500.0     # 작업 가정 (3-1): 1.5초 왕복 중 TTS 몫 — 5주차 재협상

def latency_stats(samples_ms):
    """지연 분포 요약 — 예산 판정은 평균이 아니라 p95 (3-2).
    20명 중 1명이 겪는 최악의 대기가 품질 평가를 좌우한다."""
    a = np.asarray(samples_ms, dtype=np.float64)
    return {"mean_ms": float(a.mean()), "std_ms": float(a.std(ddof=0)),
            "p95_ms": float(np.percentile(a, 95)), "n": int(a.size)}

def split_sentences(text):
    """한국어 문장 경계 분할 — 종결부호(. ? !) 뒤 공백/끝. 소수점 보호('2.5일')."""
    text = text.strip()
    if not text:
        return []
    protected = re.sub(r"(\d)\.(\d)", r"\1<DOT>\2", text)   # 소수점 보호
    parts = re.split(r"(?<=[.?!])\s+", protected)
    return [p.replace("<DOT>", ".").strip() for p in parts if p.strip()]

# ── 자가 점검 ──
s = latency_stats([120, 300, 480, 500, 900])
assert abs(s["mean_ms"] - 460.0) < 1e-6 and abs(s["p95_ms"] - 820.0) < 1e-6   # p95: np.percentile 선형 보간
assert split_sentences("불편을 드려 죄송합니다. 배송은 2.5일 걸립니다. 더 필요하신가요?") \
       == ["불편을 드려 죄송합니다.", "배송은 2.5일 걸립니다.", "더 필요하신가요?"]
print("지연 통계·문장 분할 검증 통과 ✅")
print(f"예: p95 {s['p95_ms']:.0f}ms vs 평균 {s['mean_ms']:.0f}ms — 예산 500ms를 p95로 보면 ❌")


In [ ]:
# ═══ 2.4 대화층 (5-S) — 응답 계약·분류기·LLM 인터페이스·가드레일 ═══
# ▶ 대화층은 TTS 위에 '무엇을 말할지'를 정제하는 층: 응답 계약 → 분류기 → LLM → 가드레일.
#   가드레일이 '금지 표현'을 막아야 안전한 음성 에이전트가 된다.
import re, time

RESPONSE_CONTRACT_KEYS = ("text", "intent", "strategy", "gen_ms", "state")
FORBIDDEN = ["무조건", "100% 보장", "전액 즉시", "법적으로", "계좌번호를 알려",
             "주민등록번호", "비밀번호를", "제가 책임지고"]              # AICC 금지 발화
MUST_INCLUDE = {"refund": ["죄송", "사과"], "greeting": ["안녕", "무엇을"],
                "closing": ["문의", "감사"]}

def make_response_record(text, intent, strategy, gen_ms, state):
    return {"text": text.strip(), "intent": intent, "strategy": strategy,
            "gen_ms": round(gen_ms, 1), "state": state}

def validate_response_contract(rec, max_chars):
    """위반 목록 반환. 빈 리스트 = 통과. (TTS 계약과 같은 문법: 고치지 않는다, 진단한다)"""
    v = []
    missing = [k for k in RESPONSE_CONTRACT_KEYS if k not in rec]
    if missing:
        v.append(f"누락된 키: {missing}")
        return v
    t = rec["text"]
    if not t:
        v.append("text 비어 있음")
    if len(t.replace(" ", "")) > max_chars:
        v.append(f"길이 {len(t.replace(' ', ''))}자 > 상한 {max_chars}자 (TTS 예산 환산)")
    hits = [w for w in FORBIDDEN if w in t]
    if hits:
        v.append(f"금지 표현 포함: {hits}")
    need = MUST_INCLUDE.get(rec["intent"], [])
    if need and not any(w in t for w in need):
        v.append(f"필수 표현 부재 (의도 {rec['intent']}: {need} 중 하나)")
    return v

RULES = {"greeting": ["안녕", "여보세요", "상담"], "order": ["주문", "구매", "결제"],
         "delivery": ["배송", "언제 와", "도착", "택배"], "refund": ["환불", "반품", "취소", "돈"],
         "closing": ["감사", "없어요", "끊을", "수고", "됐어요"]}

def classify_rule(text):
    """규칙 기반 의도 분류기 — LLM(5-S 셀 6)과 정확도×지연 대질."""
    for intent, kws in RULES.items():
        if any(k in text for k in kws):
            return intent
    return "unknown"

class MockLLM:
    """OpenAI 클라이언트와 같은 표면(chat(system,user)→str). 규칙+지연 모사.
    mock은 실물 레이아웃을 흉내 낼 뿐 실물보다 관대해선 안 된다."""
    def __init__(self, latency_s=0.35):
        self.latency_s, self.name = latency_s, "MockLLM"
    def chat(self, system, user):
        time.sleep(self.latency_s)
        if "의도" in system:
            return classify_rule(user)
        m = re.search(r"고객명[:=]\s*(\S+)", user)
        name = m.group(1).rstrip(".,") if m else "고객"
        if "환불" in user:
            return f"{name}님, 불편을 드려 정말 죄송합니다. 환불 절차를 바로 확인해 도와드리겠습니다."
        return f"{name}님, 문의 주신 내용 확인해 안내드리겠습니다."

def generate_guarded(fn, ctx, intent, state, max_chars=100, max_retries=1, llm=None):
    """생성 → 검증 → 위반 사유 붙여 재시도 1회 → 그래도 실패면 템플릿 폴백.
    LLM은 확률이므로 게이트는 출력 뒤에 선다 — '성공 선언은 검증 뒤에만'."""
    text, tried = fn(ctx), 0
    while tried <= max_retries:
        rec = make_response_record(text, intent, "guarded", 0, state)
        v = validate_response_contract(rec, max_chars)
        if not v:
            return rec, tried
        tried += 1
        if tried <= max_retries and llm is not None:
            text = llm.chat("아래 위반을 고쳐 두 문장 이내로 다시 써라: " + "; ".join(v), text)
    fallback = make_response_record("불편을 드려 죄송합니다. 상담원 연결해 드리겠습니다.",
                                    intent, "template_fallback", 0, state)
    return fallback, tried

# ── 자가 점검 ──
assert classify_rule("환불하고 싶어요") == "refund"
assert classify_rule("어 그게 그러니까") == "unknown"
for text, why in [("무조건 전액 즉시 환불을 보장합니다.", "금지"),
                  ("확인해 드릴게요. " * 15, "길이"),
                  ("네 알겠습니다.", "필수 부재")]:
    v = validate_response_contract(make_response_record(text, "refund", "llm_free", 0, "REFUND_FLOW"), 100)
    assert v, f"{why} 포착 실패"
llm = MockLLM(latency_s=0.02)
rec, retries = generate_guarded(lambda c: "죄송합니다. 환불은 100% 보장됩니다.",
                                {"name": "김민수"}, "refund", "REFUND_FLOW", llm=llm)
assert rec["strategy"] == "guarded" and not validate_response_contract(rec, 100)
print(f"가드레일 검증 통과 ✅ 재시도 {retries}회 → 최종 [{rec['strategy']}]")
print(f"   폴백의 존재가 곧 SLA — 최악에도 계약 통과 응답이 나간다.")


In [ ]:
# ═══ 2.5 스트리밍 TTS 코어 (6-S) — 프리버퍼·언더런·오버랩 시뮬레이터 ═══
# ▶ 프리버퍼로 언더런(끊김)을 막고, 문장 도착 즉시 합성(오버랩)으로 지연을 줄인다.
#   재생 타임라인 시뮬레이션으로 '언제 끊길지'를 코드로 미리 본다.
import re, time

def split_sentences(text):
    """한국어 문장 경계 분할 — 종결부호 뒤 공백/끝. 소수점 보호('2.5일')."""
    text = text.strip()
    if not text:
        return []
    protected = re.sub(r"(\d)\.(\d)", r"\1<DOT>\2", text)
    parts = re.split(r"(?<=[.?!])\s+", protected)
    return [p.replace("<DOT>", ".").strip() for p in parts if p.strip()]

def measure_stream(gen):
    """스트리밍 어댑터 프로토콜 계측: stream(text)→{'audio_sec':float} 청크.
    도착 타임라인 [(t, dur)]과 ttfb·총 소요·오디오 총량 반환."""
    t0 = time.perf_counter()
    timeline = []
    for chunk in gen:
        timeline.append((time.perf_counter() - t0, float(chunk["audio_sec"])))
    return {"timeline": timeline,
            "ttfb_sec": timeline[0][0] if timeline else None,
            "total_sec": timeline[-1][0] if timeline else None,
            "audio_sec": sum(d for _, d in timeline)}

class MockStreamEngine:
    """네이티브 스트리밍 엔진의 표면 재현 — 첫 청크 지연 + 청크당 생성 지연.
    mock은 실물 레이아웃을 흉내 낼 뿐, 실물보다 관대하지 않다."""
    def __init__(self, ttfb_s=0.30, per_chunk_gen_s=0.12, chunk_audio_s=0.50):
        self.ttfb_s, self.per_chunk_gen_s, self.chunk_audio_s = ttfb_s, per_chunk_gen_s, chunk_audio_s
    def stream(self, text):
        n = max(1, round(len(text.replace(" ", "")) / 5.0 / self.chunk_audio_s))  # ~5자/초
        time.sleep(self.ttfb_s)
        for i in range(n):
            if i:
                time.sleep(self.per_chunk_gen_s)
            yield {"audio_sec": self.chunk_audio_s}

def simulate_playback(timeline, prebuffer_s=0.0):
    """도착 타임라인[(t,dur)] + 프리버퍼 → {first_audio_s, underruns, done_s}.
    재생 시작은 누적 오디오가 프리버퍼를 채우는 첫 도착 시각. 이후 1배속 소비,
    버퍼가 비는 순간(언더런) 다음 청크 도착까지 정지."""
    if not timeline:
        return None
    acc = 0.0
    start = None
    for t, d in timeline:
        acc += d
        if acc >= prebuffer_s:
            start = t
            break
    if start is None:
        start = timeline[-1][0]
    underruns, clock = [], start
    for t, d in timeline:
        if t <= start:
            continue                    # 이미 버퍼에 모인 청크
        if clock < t:
            underruns.append(t)         # 재생할 것이 없어 대기
            clock = t
        clock += d                      # 1배속 소비
    return {"first_audio_s": start, "underruns": underruns, "done_s": clock,
            "num_underruns": len(underruns)}

def min_prebuffer(timeline, grid=None):
    """언더런 0회가 되는 최소 프리버퍼(초)를 격자 탐색으로 도출 — 정책 상수도 실측에서."""
    grid = grid or [round(0.1 * i, 1) for i in range(0, 41)]
    for pb in grid:
        r = simulate_playback(timeline, prebuffer_s=pb)
        if r and not r["underruns"]:
            return pb, r
    return None, None

class MockStreamLLM:
    """토큰 단위로 흘러나오는 LLM의 표면 — OpenAI stream=True와 같은 소비 패턴."""
    def __init__(self, per_token_s=0.03):
        self.per_token_s = per_token_s
    def stream_tokens(self, system, user):
        answer = ("불편을 드려 정말 죄송합니다. 주문 내역을 확인해 보니 배송이 지연되고 있습니다. "
                  "바로 환불 처리 도와드리겠습니다.")
        for tok in re.findall(r"\S+\s*", answer):
            time.sleep(self.per_token_s)
            yield tok

def overlap_pipeline(llm, tts_latency_fn):
    """토큰 스트림을 받으며 문장 경계마다 즉시 TTS 디스패치(오버랩).
    생성이 끝나기 전에 합성이 시작됨 → 체감 첫 소리가 앞당겨진다."""
    t0 = time.perf_counter()
    buf, events, tts_free_at = "", [], 0.0
    def flush(sent):
        nonlocal tts_free_at
        ready = time.perf_counter() - t0
        start = max(ready, tts_free_at)          # TTS는 단일 워커 가정(직렬)
        done = start + tts_latency_fn(sent)
        tts_free_at = done
        events.append({"sent": sent, "ready": ready, "tts_done": done})
    for tok in llm.stream_tokens("콜센터 상담원", "환불 요청"):
        buf += tok
        sents = split_sentences(buf)
        if len(sents) > 1:
            for s in sents[:-1]:
                flush(s)
            buf = sents[-1]
    if buf.strip():
        flush(buf.strip())
    return {"events": events,
            "first_audio_sec": events[0]["tts_done"] if events else None,
            "all_done_sec": events[-1]["tts_done"] if events else None}

# ── 자가 점검 ──
tl = [(0.0, 0.5), (1.0, 1.0)]
assert simulate_playback(tl, prebuffer_s=0.0)["num_underruns"] == 1   # 0.5s 공백
pb, rb = min_prebuffer(tl)
assert rb is not None and rb["num_underruns"] == 0
print(f"min_prebuffer 검증 ✅ 언더런 0의 최소 프리버퍼 = {pb}s "
      f"(첫 소리 {rb['first_audio_s']:.1f}s — 매끄러움은 공짜가 아니다)")
llm2 = MockStreamLLM(per_token_s=0.0)
res = overlap_pipeline(llm2, lambda s: 0.5)
assert res["first_audio_sec"] is not None and len(res["events"]) >= 2
print(f"오버랩 검증 ✅ 문장 {len(res['events'])}개, 첫 소리 {res['first_audio_sec']:.2f}s")


In [ ]:
# ═══ 2.6 macOS(Apple Silicon) 실행 어댑터 (신규) 🍎 ═══
# ▶ Apple Silicon에서 TTS 실행: Colab GPU 없이도 로컬로. 계약은 그대로.
# 정의만 하는 셀 — import는 호출 시점에만 (Colab 환경과 무관하게 로드됨)

def mac_melo_synthesize(text, language="KR", device="cpu", speed=1.0):
    """MeloTTS — MIT, 한국어(KR), CPU 실시간 + MPS 지원 (3-L의 검증 대상).
    설치: pip install melotts   →   모델: TTS(language='KR', device='cpu')
    CLI: melotts <text> out.wav -l KR -d mps"""
    from melo.api import TTS
    model = TTS(language=language, device=device)
    sid = model.hps.data.spk2id[language]
    return model.tts_to_file(text, sid, "mac_melo.wav", speed=speed)

def mac_supertonic_synthesize(text, total_steps=8, speed=1.0):
    """Supertonic 3 — 31개 언어(한국어 포함), ONNX CPU 실시간, MIT (3-2의 엔진).
    sherpa-onnx에서도 동일 모델을 int8로 로드 가능 (7개 서브모델 구성)."""
    from supertonic import Supertonic
    model = Supertonic()
    return model.synthesize(text, total_steps=total_steps, speed=speed)

def mac_sherpa_onnx_synthesize(model_config, text, sid=0, speed=1.0):
    """sherpa-onnx(ONNX 통합 런타임) — macOS·엣지. Supertonic/Kokoro/Piper/MeloTTS를
    한 API로. 설치: pip install sherpa-onnx. 스트리밍 TTS도 지원.
    model_config 예: sherpa_onnx.OfflineTtsConfig(..., supertonic=..., kokoro=..., vits=...)"""
    import sherpa_onnx
    tts = sherpa_onnx.OfflineTts(model_config)
    audio = tts.generate(text, sid=sid, speed=speed)
    return audio.samples, audio.sample_rate   # (ndarray, sr) — 계약 어댑터로 16kHz 규격화

print("macOS 어댑터 정의 완료 ✅ (import는 호출 시점에만 — 무설치)")
print("추천: sherpa-onnx로 Supertonic int8 + Kokoro를 한 API에서 — 한국어는 Supertonic")


## 2.7 📄 왕복 CER 표준 장비 (3-F ~ 3-ST) — 합성음은 ASR이 되받아 적어 심사한다

> "지원은 귀가 아니라 저울로 판정한다." — TTS가 만든 소리를 2주차 faster-whisper가 받아 적고, 원문과 CER을 잽니다. (TTS↔ASR 왕복 = 파이프라인이 자기 자신을 시험)

| 함수 | 역할 | GPU |
|---|---|---|
| `cer(ref, hyp)` | 2-1 표준 — 왕복 판정의 눈금 | ❌ |
| `make_tts_record(text, uid, audio, sr, latency_ms, engine)` | 합성 결과를 계약 dict로 (validate_tts_contract 통과) | ❌ |
| `roundtrip_cer(text, audio, sr)` | 합성음 → faster-whisper → 원문 CER (판정 셀) | ✅ |
| `free_gpu()` | `del` 후 GPU 캐시 반환 (인라인 del + free_gpu 관례) | — |

**판정 원칙 (3-F)**: 코드에 언어 테이블이 있으면 읽고(3-K), 없으면 합성→역전사→CER로 잰다(3-F). **코드가 침묵하면 실험이 판정한다.**

## 2.8 📄 엔진 9종 통합 카드 표 (Week 3 재협상 테이블)

| 엔진 | 한국어 | 지연 vs 0.5s | 라이선스 | macOS | sr 위반 | 한 줄 평 |
|---|---|---|---|---|---|---|
| Supertonic 3 (3-2) | ✅ | ✅ (CPU 실시간) | MIT | ✅ | 44.1k | 경량 실무 표준 |
| MeloTTS (3-L) | ✅ (테이블) | ✅ (CPU 실시간 검증) | MIT | ✅ | 44.1k | VITS·핀 우회 필수 |
| XTTS v2 (3-3) | ✅ | ⚠️ T4 GPU | CPML ❌ | ✖ | 24k | 품질 상한선 (비상업) |
| Kokoro (3-K) | ❌ (9개 언어) | ✅ | Apache | ✅ | 24k | 테이블 판정 |
| MMS-TTS (3-S) | ✅ (kor) | ✅ (VITS) | CC-NC ❌ | ✅ | **16k 일치** | 저장소=지원, uroman |
| S1-mini (3-F) | ✅ (실측) | ⚠️ AR | CC-NC ❌ | ✖ | 44.1k | 이중 게이트 |
| MOSS v1.5 (3-M) | ✅ (실측) | ⚠️ AR | Apache | ✖ | 24k | 3축 모두 통과 최초 |
| Qwen3-TTS (3-Q) | ✅ | ⚠️ AR | Apache | ✖ | 24k | device-side assert |
| GPT-SoVITS (3-G) | ✅ (v2+) | ⚠️ 혼혈 | MIT | ✖ | 32k/24k/48k | sr는 속성 |
| F5-TTS-ko (3-D) | ✅ (실측) | ⚠️ flow | Apache△(데이터 미공개) | ✖ | — | 체크포인트 부검 |
| StyleTTS2 (3-ST) | ❌ (부품) | ✅ (NAR) | MIT+GPL | ✖ | — | Kokoro의 조상 |

> **3주차 결론 (3-M)**: 세 축을 동시에 만족하는 카드는 없었다 — '한국어✅·예산❌·Apache✅'(Qwen3) · '한국어❌·예산✅·Apache✅'(Kokoro) · '한국어✅·예산✅·NC❌'(S1). **MOSS v1.5가 첫 3축 후보.** 지연 축의 진짜 해법은 6주차 스트리밍.

## 2.9 📄 엔진별 어댑터 요약 — 계약 위반 6종 유형

| 유형 | 사례 | 대응 |
|---|---|---|
| sr 위반 (24k/44.1k → 16k) | 거의 전 엔진 | 리샘플(`librosa.resample`) — **변환 책임은 파이프라인** |
| dtype 위반 (float32 → PCM_16) | 전부 | 캐스팅/양자화 (MMS는 캐스팅만으로 충분) |
| 반환 타입 (파일/리스트/ndarray) | gTTS/XTTS/Supertonic | 어댑터가 통일 |
| **입력측 계약** (로마자화) | MMS uroman | 입력 전처리 |
| 신뢰도 없음 | 대부분 | 계약에 logprob 없음 — TTS는 신뢰도 개념이 약함 |

> **계약 위반은 쌍방향이다 (3-Q)**: 모델은 우리의 16kHz int16을 모르고, 우리는 모델의 화자·언어·정밀도 요구를 모른 채 출발한다 — 양쪽 계약 모두 실물(설치본·모델 자신)에서 읽어라.


## 2.8 [REAL] 실물 실행 — MeloTTS · sherpa-onnx (Supertonic/Kokoro) 🍎

> 2.6의 macOS 어댑터를 **실제 호출**해 한국어/영어 문장을 합성하고, **TTS 계약(16k mono f32)** 통과를
> 실물로 확인합니다. 첫 실행 시 모델 다운로드. 준비: `bash setup_apple_silicon.sh tts`


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd() if (Path.cwd() / "aicc_env.py").exists() else Path.cwd().parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import aicc_env as ae

TEXT_KO = "안녕하세요, 지난달 요금이 평소보다 많이 나온 것 같아서 확인 부탁드립니다."

def _contract(audio, sr, text, engine):
    # 2.2 TTS 계약 키: audio·sr·duration_ms·latency_ms·engine — 16k mono f32 규격화
    import numpy as np
    audio = ae.to_16k_mono(audio, sr)
    return {"audio": audio, "sr": 16000,
            "duration_ms": round(len(audio) / 16000.0 * 1000.0, 1),
            "latency_ms": 0.0, "engine": engine}

# ① sherpa-onnx Supertonic(ko) — cp314 지원, 한국어 31언어 모델 (우선 엔진, ~130MB 1회 다운로드)
if ae.has("sherpa_onnx"):
    import numpy as np
    out, ms = ae.timed(ae.synth_korean, TEXT_KO)
    assert out is not None, "Supertonic 합성 실패"
    audio, sr = out
    rec = _contract(audio, sr, TEXT_KO, "supertonic-ko")
    assert validate_tts_contract(rec)
    ae.save_wav(ae.HERE / "assets" / "tts_supertonic_ko.wav", audio, sr)
    print(f"Supertonic(ko) 합성 {ms:.0f}ms | 원본 sr {sr} | 계약 통과 ✅")
else:
    print("sherpa-onnx 미설치 → 스킵.  bash setup_apple_silicon.sh tts")

# ② MeloTTS-KR — Python 3.12/3.13 환경에서만 설치 가능 (fugashi 의존성)
if ae.has("melotts"):
    import numpy as np
    wav_path, ms = ae.timed(mac_melo_synthesize, TEXT_KO, language="KR", device="cpu", speed=1.0)
    audio, sr = ae.load_wav(wav_path)
    rec = _contract(audio, sr, TEXT_KO, "melotts-KR")
    assert validate_tts_contract(rec)
    ae.save_wav(ae.HERE / "assets" / "tts_melo_ko.wav", audio, 16000)
    print(f"MeloTTS-KR 합성 {ms:.0f}ms | 계약 통과 ✅")
else:
    print("melotts 미설치 → 스킵 (한국어는 Supertonic이 대체)")


# 3. 실험 진행 방법 🧪

## 3-1. 표준 워크플로우 (모든 노트북 공통 7단계)

```
① 설치 (설계된 재시작 준비)  → ② 폰트·시드  → ③ 자산 복원(계약·정규화기·발화)
   → ④ 엔진 로드·계약 검증(예측 먼저)  → ⑤ 합성 + 변환 파이프라인  → ⑥ 역전사 CER·지연 실측
   → ⑦ 라이선스 판정·카드 기록 + del/free_gpu
```

> ⚠️ **설치 셀 최우선·1회**. METADATA를 먼저 읽어 재시작을 '절차'로 설계(3-F) — 커널 교체형(3-G)은 배너가 정상 동작.
> 계약 검증은 **예측부터** (3-M): "몇 건이, 어느 키로 위반될까?" — 예측이 깨지면 통계가 갱신된다(3-S).

## 3-2. 노트북별 실험 가이드

| 노트북 | 실험 목적 | 핵심 절차 | 판단 기준 |
|---|---|---|---|
| **3-1** | 3단 구조 해부 | 정규화 → G2P → 멜 관찰 → Griffin-Lim → 계약·예산 | 정규화 A/B 청취, 위상 손실을 귀로 |
| **3-2** | 경량 로컬 | Supertonic 로드 → 44.1k 계약 방어 → gTTS vs 로컬 p95 → steps/speed → 내장 정규화 A/B → 문장 분할 TTFB | **p95 ≤ 500ms**, TTFB 개선 배수 |
| **3-3** | 고품질·다국어 | CPML 동의 → 어댑터(24k 리스트) → 4개 언어 스윕 → 한영 음차 → 총비용 | 라이선스 판정 + 언어별 편차 |
| 3-F | S1-mini | 이중 게이트 → 코드 침묵 확인 → 합성 → **역전사 CER** → 160/441 변환 | 왕복 CER 판정 |
| 3-L | MeloTTS | 핀 우회 → 'KR' 테이블 → CPU 실시간 **GPU/CPU 이중 실측** | RTF<1 + 0.5s 구분 해석 |
| 3-K | Kokoro | LANG_CODES → 9개 언어 전수(assert 보증) → 변환 → 지연 | 전수 assert + 언어별 지연 |
| 3-M | MOSS | 설치 전략 측정 → 언어·지연·라이선스 3축 | 3축 판정 도구 매칭 |
| 3-Q | Qwen3-TTS | 소스 먼저 읽기 → device-side assert 사건 → 계약 → 지연 | 재시작 여부부터 진단 |
| 3-S | MMS | kor 저장소 → uroman → **sr 일치(1/7)** → 캐스팅만 변환 | 입력 계약 발견 |
| 3-G | GPT-SoVITS | condacolab 커널 교체 → sr를 속성으로 → 혼혈 지연 곡선 | 버전 각주 판정 |
| 3-D | F5-ko | vocab 부검 → 체크포인트 부검 → 포맷 어댑터 → 왕복 CER | 저울(3-D와 동일 눈금) |
| 3-ST | StyleTTS2 | 스타일 확산(주사위) → 한국어 기각 → 부품 지층 | diffusion_steps=5로 충분 |
| **5-S** | 대화층 | 응답 계약 → 분류기 2종 → 3전략 → 가드레일 → 예산 통합 | 분류+생성 합 ≤ 0.5s |
| **6-S** | 스트리밍 | 분할/프로토콜/재생 시뮬레이터 → 프리버퍼 실측 → 오버랩 | 언더런 0의 최소 프리버퍼 |

## 3-3. macOS에서 실습하는 법 (신규) 🍎

| 노트북 | Mac에서의 진행 방법 |
|---|---|
| 3-1 | gTTS(클라우드)/Mock 그대로 — 코드 수정 없음. 정규화·G2P·보코더 실습 전부 실행 가능 |
| 3-2 | Supertonic(ONNX CPU) 그대로 — **3-2는 애초에 CPU 세션**. p95·steps·TTFB 실습 무수정 |
| 3-3 | XTTS는 비상업(CPML)이라 평가용으로만. macOS에선 대신 MeloTTS로 다국어/음차 실습 |
| 3-L / 3-K / 3-S | CPU 실시간/완주 세션 — 코드 수정 없음 (3-L은 `device='mps'`로 가속 가능) |
| 5-S / 6-S | 순수 시뮬레이션 — 무수정. 6-S의 MockStreamEngine 그대로 |
| 3-F/3-M/3-Q/3-G/3-D/3-ST | T4 전용(무거움·비상업) — Colab 권장. macOS에선 결론(카드 표)만 학습 |
| **맥 전용 실험** | sherpa-onnx로 Supertonic int8 vs MeloTTS(MPS) vs gTTS를 **같은 계약·p95 예산**으로 재비교 — 2-2 토너먼트의 TTS판 |

## 3-4. 실험 판단 기준 모음 (빠른 참조)

| 지표 | 목표값 | 출처 |
|---|---|---|
| TTS 지연 (p95) | ≤ 500ms (1.5s 중 TTS 몫) | 3-1/3-2 |
| 역전사 CER | 도메인별 목표 (합성음이 낮게 나오는 게 정상) | 3-F |
| RTF | < 1.0 (실시간 후보) | 3-L |
| TTFB | 문장 분할로 배수 개선 (일괄 vs 분할 비교) | 3-2 |
| 대화층 예산 | 분류 + 생성 합 ≤ 0.5s | 5-S |
| 스트리밍 | 언더런 0의 **최소 프리버퍼** (감이 아니라 실측) | 6-S |
| 3축 판정 | 한국어=실험 / 지연=측정 / 라이선스=문서 | 3-M |


# 4. 효율적 설계를 위한 아키텍처 🏛️

## 4-1. TTS 계약이 모든 것을 지탱한다 (ASR 계약과 대칭)

```
[엔진 11종: gTTS·Supertonic·XTTS·Qwen3·MeloTTS·MOSS·S1·Kokoro·MMS·GPT-SoVITS·F5]
   └── 어댑터(sr/dtype/타입 변환) ──▶ {"audio":16k mono, "sr":16000,
                                      "duration_ms", "latency_ms", "engine"}
                                          │
                                          ▼
                    5-S 대화층 출력(text) → 6-S 재생 버퍼 · 전화망 전송
```

> **계약은 어길 때 존재를 증명한다** (3-D): 44.1kHz가 경계에서 차단되지 않았다면 2.76배 느린 목소리가 하류로 조용히 흘러갔을 것.

## 4-2. 파이프라인 (텍스트 → 계약 오디오)

```
LLM 응답(5-S) ──▶ normalize_aicc(전화→금액→개수) ──▶ G2P(발음 치유)
   ──▶ [엔진 어댑터: 입력 계약(MMS uroman) + 출력 변환(16k/PCM_16)]
   ──▶ 계약 dict ──▶ validate_tts_contract ──▶ 재생/전송
```

## 4-3. 문장 분할 유사-스트리밍 — 비스트리밍 엔진의 구제 (3-2)

```
일괄: [————— 전체 합성 —————] ▶ 재생              TTFB = 전체 지연
분할: [1문장] ▶ 재생 [2문장] [3문장]               TTFB = 첫 문장 지연
```
총 합성 시간은 오히려 늘지만(오버헤드), 고객이 느끼는 것은 총량이 아니라 **첫 소리까지의 침묵**. (2-R의 발화-처리 중첩과 같은 원리의 두 번째 적용)

## 4-4. 스트리밍 TTS — 첫 소리와 매끄러움의 거래 (6-S)

```
엔진 스트림 ─▶ 도착 타임라인 ─▶ [재생 커서 시뮬레이션]
                 │                 └─ 프리버퍼 ↑ → 언더런 0, 첫 소리 ↓
                 └─ 정책 상수는 실측에서: min_prebuffer(timeline)

LLM 토큰 스트림 ─▶ 문장 경계마다 즉시 TTS 디스패치(오버랩) ─▶ 체감 첫 소리 단축
```
스트리밍은 **엔진 기능이 아니라 설계 계층** — 네이티브 스트림이 없어도 문장 분할이 구제한다.

## 4-5. 대화층 (5-S) — 예산은 층마다 상속된다

```
1.5s 왕복 = ASR 0.5 + TTS 0.5 + 대화층 0.5
대화층 0.5s = 의도 분류 + 응답 생성 (각자 다시 예산 분할)

응답 3전략:  템플릿(~0ms, 100% 재현) vs LLM 자유(왕복, 유연) vs 하이브리드(템플릿+슬롯)
가드레일:   생성 → validate_response_contract → 위반 시 재시도 1회 → 템플릿 폴백 (SLA)
```

## 4-6. macOS 배포 아키텍처 (신규) 🍎

```
[로컬 온디바이스]                  [로컬 고속]                     [클라우드/API]
Supertonic 3 (ONNX, MIT)        sherpa-onnx (통합 런타임)       gTTS (클라우드)
MeloTTS (MPS, MIT)              Supertonic int8 / Kokoro 등        │
      │                              │                        지연 불안정(네트워크)
  배터리·오프라인 우선             실시간·스트리밍 대응          → p95 예산 위험
```

| 선택 | 적합 시나리오 | 고려사항 |
|---|---|---|
| Supertonic 3 | 오프라인 실시간, 한국어, MIT — AICC 기본 | 44.1k → 계약 어댑터 필수 |
| MeloTTS | MPS 가속, VITS, MIT | 44.1k, 핀 우회 필요 |
| sherpa-onnx | macOS·엣지 통합, 스트리밍 | 모델별 라이선스 확인 |
| gTTS | 품질·즉시 사용 | 비용·네트워크·p95 변동 |

## 4-7. 설계 원칙 요약 (이 5줄이 3주차의 전부)

1. **계약은 최소 보장, 확장은 자유** — `TTS_CONTRACT_KEYS` 5개만 검사, 엔진은 어댑터 뒤로 격리.
2. **변환 책임은 파이프라인 쪽** — 엔진이 안 고쳐지면 파이프라인이 규격화한다 (sr/dtype/타입).
3. **축마다 판정 도구가 다르다** — 라이선스=문서, 한국어=실험(왕복 CER), 지연=측정. 어떤 축도 다른 도구를 빌릴 수 없다 (3-M).
4. **예산은 층마다 상속된다** — 1.5s가 ASR/TTS/대화층으로, 또 그 안에서 분류/생성으로. 판정은 평균이 아니라 **p95**.
5. **성공 선언은 검증 뒤에만** — LLM은 확률이므로 가드레일(응답 계약)·재시도·템플릿 폴백이 SLA를 지킨다 (5-S). 매끄러움(언더런 0)도 공짜가 아니라 최소 프리버퍼로 산다 (6-S).
